# NB_lab_test_trends

In [0]:
import time

_run_start = time.time()
target_table = "adb_healthcare.gold.lab_test_trends"
print("=" * 70)
print(f"[START] NB_lab_test_trends building '{target_table}'")

In [0]:
%sql
CREATE OR REPLACE TABLE adb_healthcare.gold.lab_test_trends
COMMENT 'Gold: one row per test type per month with abnormal/critical rates and month-over-month trend (STTM silver to gold)'
AS
WITH lab_results_dedup AS (
  -- silver.lab_results contains every test once per load batch: keep the latest version of each test
  SELECT *
  FROM adb_healthcare.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY landing_timestamp DESC, update_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    year(test_date)                                                   AS year,
    month(test_date)                                                  AS month,
    CAST(date_format(date_trunc('MONTH', test_date), 'yyyyMMdd') AS INT) AS month_date_key,
    count(*)                                                          AS total_tests,
    count_if(result_status = 'Abnormal')                              AS abnormal_test_count,
    count_if(result_status = 'Critical')                              AS critical_test_count
  FROM lab_results_dedup
  GROUP BY test_name, year(test_date), month(test_date), date_trunc('MONTH', test_date)
),
with_rates AS (
  SELECT
    m.*,
    CAST(round(m.abnormal_test_count / m.total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(round(m.critical_test_count / m.total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    lag(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM with_rates r
)
SELECT
  w.test_name                                                            AS test_name,
  CAST(w.year AS INT)                                                    AS year,
  CAST(w.month AS INT)                                                   AS month,
  c.month_name                                                           AS month_name,    -- e.g. March
  c.quarter_name                                                         AS quarter_name,  -- e.g. Q1
  CAST(w.total_tests AS INT)                                             AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                     AS critical_test_count,
  w.abnormal_rate_pct                                                    AS abnormal_rate_pct,
  w.critical_rate_pct                                                    AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                   AS prev_month_abnormal_rate_pct,  -- NULL for a test's first month
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct  -- percentage-point change, NULL if no prior month
FROM with_lag w
LEFT JOIN adb_healthcare.silver.calendar c
  ON c.date_key = w.month_date_key;

In [0]:
records_count = spark.table(target_table).count()
print(f"[EXIT] Returning records_count = {records_count} to caller")
print(f"[END] NB_lab_test_trends finished for table '{target_table}' in {time.time() - _run_start:.1f}s")
print("=" * 70)
dbutils.notebook.exit(str(records_count))